### Sinaliza Equipamentos

- O objetivo desse script é gerar uma sinalização dnp3 massiva para relés SEL para o SCADA variando o mapa dnp3 \
- Nesse exemplo, primeiro encontrou-se a posição da LT03 no mapa dnp3 e depois variou-se o valor desse ponto.

In [ ]:
import pandas as pd 
from SELFtools import Rele,ReleFTP,Arquivo
import itertools
from time import sleep

In [3]:
df = pd.DataFrame({'SE': ['LAB', 'LAB2', 'LAB3'],
                   'EQP': ['EQP1', 'EQP2', 'EQP3'],
                   'IP': ['10.10.10.1', '10.10.10.2', '10.10.10.3']})
df = df.sort_values(by = ['SE'],ascending=False)
df.head()

,SE,EQP,IP
2,LAB3,EQP3,10.10.10.3
1,LAB2,EQP2,10.10.10.2
0,LAB,EQP1,10.10.10.1


In [4]:
def valida_lt03(arquivo):
    LT03 = arquivo.verificar_parametro('SET03')
    if 'LOCAL' in LT03:
        return True
    else:
        print(f'LT03 tem valor {LT03}')
        return False

In [ ]:
#for rele,dfx in df.iloc[0:1].iterrows():
for rele,dfx in df.iterrows():
    print(f'Iniciando conexão com {dfx.EQP} da SE {dfx.SE} de ip : {dfx.IP}')
    rele = ReleFTP(dfx.IP,'2AC',['Senha1','Senha2','Senha3'],dfx.SE,dfx.EQP)
    print('Baixando arquivo de lógicas')
    rele.baixar_arquivos_ftp('SET_L1.TXT')
    sleep(2)
    arquivo = rele.abrir_arquivo(f'Enviado/{dfx.SE}_{dfx.EQP}_SET_L1__atualizado.TXT')
    if(valida_lt03(arquivo)):
        print('Baixando arquivo de binárias')
        rele.baixar_arquivos_ftp('SET_D1.TXT')
        sleep(2)
        arquivo_atualizado = f'Enviado/{dfx.SE}_{dfx.EQP}_SET_D1__atualizado.TXT'
        arquivo = rele.abrir_arquivo(arquivo_atualizado)
        print('Fazendo Backup')
        backup = arquivo.fazer_backup()
        BI = arquivo.verificar_uso_variavel('LT03')[0]
        arquivo.atualizar_parametro(f'{BI}','1')
        print('Enviando Arquivo')
        rele.enviar_arquivo_atualizado(backup,arquivo_atualizado,'SET_D1.TXT')
        sleep(20)
        print('Enviando Backup')
        rele.enviar_backup(backup,'SET_D1.TXT')
        sleep(20)
        rele.fechar_conexao()
    else:
        print('LOCAL não está comentado na LT03')
        continue

#### Output : 
Iniciando conexão com EQP1 da SE  LAB de ip : 10.10.10.1 \
220 \
Baixando arquivo de lógicas \
Baixando arquivo de binárias \
Fazendo Backup \
Enviando Arquivo \
RETR SET_D1.TXT \
Diferenças após ajuste: 0 \
As alterações foram enviadas com sucesso \
Enviando Backup \
As alterações foram enviadas com sucesso \
Conexão FTP fechada \
Iniciando conexão com EQP2 da SE LAB2 de ip : 10.10.10.2 \
220 \
Baixando arquivo de lógicas \
Baixando arquivo de binárias \
Fazendo Backup \
Enviando Arquivo \
RETR SET_D1.TXT \
Diferenças após ajuste: 0 \
As alterações foram enviadas com sucesso \
Enviando Backup \
As alterações do 10.10.10.2 foram enviadas \
Conexão FTP fechada \
Iniciando conexão com EQP3 da SE LAB3 de ip : 10.10.10.3 \
... \
Diferenças após ajuste: 0 \
As alterações foram enviadas com sucesso \
Enviando Backup \
As alterações do 10.10.10.3 foram enviadas \